# Phase 8 Swin MoE Kaggle GPU Migration
This notebook trains and evaluates three Swin Transformer configurations (Vanilla, Spatially-Aware MoE, Warm-Init Spatially-Aware MoE) on the BUSI dataset, followed by zero-shot evaluation on BUS-BRA.

**Ensure you have a GPU accelerator enabled in your Kaggle session.**

## 1. Environment Setup

In [ ]:
import torch
import sys
import os

print('CUDA Available:', torch.cuda.is_available())
print('Device Name:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

assert torch.cuda.is_available(), 'CUDA is not available! Please enable GPU in Kaggle settings.'


## 2. Repository Setup
Clone the repository containing the Phase 8 models and metrics.

In [ ]:
!git clone https://github.com/toqeer-ahmed/vision_tranformer_moe.git vision_transformer_research
import os
import sys
os.chdir('vision_transformer_research')
sys.path.append(os.path.abspath('.'))

!pip install -r requirements.txt


## 3. Dataset Setup
Assuming BUSI is mounted at `../input/busi-dataset` and BUS-BRA at `../input/busbra-dataset`.
Update the config to point to the Kaggle paths.

In [ ]:
# Please manually upload BOTH datasets to Kaggle!
# We have removed the auto-download code because the Kaggle datasets were deleted.


In [ ]:
import os
import yaml

def find_dataset_path(base_dir, target_folder_name):
    print(f"Searching for {target_folder_name} in {base_dir}...")
    for root, dirs, files in os.walk(base_dir):
        if target_folder_name in dirs:
            found_path = os.path.join(root, target_folder_name)
            print(f"Found at: {found_path}")
            return found_path
    print(f"WARNING: Could not find {target_folder_name}!")
    return None

# Dynamically find the datasets regardless of how Kaggle extracted the zip folders (archive / archive 1)
busi_path = find_dataset_path("/kaggle/input", "Dataset_BUSI_with_GT")
busbra_path = find_dataset_path("/kaggle/input", "BUSBRA")

# Fallback just in case
if not busi_path: busi_path = "/kaggle/input/busi-dataset/Dataset_BUSI_with_GT"
if not busbra_path: busbra_path = "/kaggle/input/busbra-dataset/BUSBRA"



## 4. Train Vanilla Swin

In [ ]:
!python training/train_swin_vanilla.py --config configs/swin_segmentation.yaml


## 5. Train Spatially-Aware Swin-MoE

In [ ]:
!python training/train_swin_moe.py --config configs/swin_segmentation.yaml


## 6. Train Warm-Initialized Spatially-Aware Swin-MoE

In [ ]:
!python training/train_swin_moe.py --config configs/swin_segmentation.yaml --warm-init


## 7. Evaluate on BUSI Test Set
The training scripts automatically evaluate on the test set and output `test_metrics.json` in their log directories.

## 8. Zero-Shot Evaluation on BUS-BRA

In [ ]:
import glob

# Find best checkpoints automatically
vanilla_dirs = sorted(glob.glob('outputs/swin_vanilla_*'))
moe_dirs = sorted(glob.glob('outputs/swin_moe_*'))

# NOTE: Update these paths based on the actual timestamped folders generated above
vanilla_ckpt = os.path.join(vanilla_dirs[-1], 'checkpoints', 'best_model.pth') if vanilla_dirs else ''
moe_ckpt = os.path.join(moe_dirs[0], 'checkpoints', 'best_model.pth') if moe_dirs else ''
warm_moe_ckpt = os.path.join(moe_dirs[-1], 'checkpoints', 'best_model.pth') if len(moe_dirs) > 1 else ''

print(f"Evaluating checkpoints:\nVanilla: {vanilla_ckpt}\nMoE: {moe_ckpt}\nWarm MoE: {warm_moe_ckpt}")

evaluate_cmd = f"python scripts/evaluate_swin.py" # Note: evaluate_swin.py internally hardcodes checkpoints. Let's run a custom block instead to pass the dynamic paths.

In [ ]:
from scripts.evaluate_zero_shot import evaluate_zero_shot

checkpoints = {
    "swin_vanilla": {
        "config": "configs/swin_segmentation.yaml",
        "checkpoint": vanilla_ckpt
    },
    "swin_moe_spatial": {
        "config": "configs/swin_segmentation.yaml",
        "checkpoint": moe_ckpt
    },
    "swin_moe_warm_init": {
        "config": "configs/swin_segmentation.yaml",
        "checkpoint": warm_moe_ckpt
    }
}

output_dir = 'outputs/swin_generalization_results'
os.makedirs(output_dir, exist_ok=True)

if all(os.path.exists(c['checkpoint']) for c in checkpoints.values()):
    evaluate_zero_shot(busbra_path, checkpoints, output_dir)
else:
    print("Error: One or more checkpoints are missing! Ensure all 3 models finished training.")


## 9. Visualizations & 10. Final Comparison Report
Collect all results and generate `PHASE8_SWIN_RESULTS.md`.

In [ ]:
import json

report_lines = [
    "# Phase 8: Swin Transformer & MoE Experiments on Kaggle GPU",
    "",
    "## 1. Experimental Setup",
    "- **Dataset**: BUSI (Train/Val/Test) -> BUS-BRA (Zero-Shot)",
    "- **Models**: Vanilla Swin-Tiny, Spatially-Aware Swin-MoE, Warm-Init Swin-MoE",
    "- **Framework**: PyTorch AMP (Mixed Precision)",
    "",
    "## 2. BUSI Test Results",
]

# Collect BUSI Metrics (from training logs)
for name, dirs in [('Vanilla', vanilla_dirs), ('MoE', [moe_dirs[0]] if moe_dirs else []), ('Warm MoE', [moe_dirs[-1]] if len(moe_dirs)>1 else [])]:
    if dirs and os.path.exists(os.path.join(dirs[-1], 'logs', 'test_metrics.json')):
        with open(os.path.join(dirs[-1], 'logs', 'test_metrics.json'), 'r') as f:
            m = json.load(f)
            report_lines.append(f"**{name}**: mIoU {m['mean_iou']:.4f} | mDice {m['mean_dice']:.4f} | Precision {m['mean_precision']:.4f} | Recall {m['mean_recall']:.4f}")

report_lines.extend([
    "",
    "## 3. BUS-BRA Zero-Shot Results",
])

if os.path.exists(os.path.join(output_dir, 'zero_shot_metrics.json')):
    with open(os.path.join(output_dir, 'zero_shot_metrics.json'), 'r') as f:
        zs = json.load(f)
        for k, v in zs.items():
            report_lines.append(f"**{k}**: mIoU {v['mean_iou']:.4f} | mDice {v['mean_dice']:.4f} | Precision {v.get('mean_precision', 0):.4f} | Recall {v.get('mean_recall', 0):.4f}")

with open('outputs/PHASE8_SWIN_RESULTS.md', 'w') as f:
    f.write('\n'.join(report_lines))

print("Final Report Generated at outputs/PHASE8_SWIN_RESULTS.md")
